# Normalisation and shrinkage

*Question, Intuition, Math, Code, Assumptions, How it breaks*

## 1. Question

Ronaldo (The Real One) scored 25 league goals in 2002-03. Haaland scored 36 in 2022-23. Which
is the better season?

The counts are not comparable. They were scored in different leagues, in
different decades, against different defences, under different rules about what
even counts as a foul. Before any ranking can pool them they have to go onto one
scale, and the choice of scale decides the answer.

## 2. Intuition

Stop asking *how many* and start asking *how far ahead of everyone else*.

A player who scores 25 in a season where the best forwards score 20 has done
something more impressive than one who scores 25 in a season where they score
35. The comparison that survives across eras is not the raw number. It is the
**distance from the crowd**, measured in units of how spread out that crowd is.

That is a standard score, and it is the whole idea. Everything after this is
detail, except that two of those details change the answer, so they get their
own sections below.

## 3. Math

For player $i$ in league-season $g$, with raw rate $x_{ig}$:

$$z_{ig} = \frac{x_{ig} - \mu_g}{\sigma_g}$$

where $\mu_g$ and $\sigma_g$ are the mean and standard deviation of *everyone
who played in that league that season*.

**The denominator is a choice.** I use the population standard deviation
($\text{ddof}=0$), not the sample one ($\text{ddof}=1$). The sample correction
exists to estimate the spread of a larger population from a subset, and here
there is no larger population. The players who played the 2003-04 Premier League
*are* the 2003-04 Premier League. Dividing by $n-1$ would be correcting for a
sampling step that never happened.

With $n \approx 500$ per group the numerical difference is tiny. The reason to
get it right is that the alternative is not defensible if somebody asks.

In [1]:
import warnings

import matplotlib
import pandas as pd

warnings.filterwarnings("ignore")
matplotlib.rcParams["figure.figsize"] = (10, 5.5)

SAMPLE = "../data/sample"
ranking = pd.read_parquet(f"{SAMPLE}/ranking.parquet")
seasons = pd.read_parquet(f"{SAMPLE}/player_season_scored.parquet")
offsets = pd.read_parquet(f"{SAMPLE}/league_offsets.parquet")

Here is the comparison this chapter opened with, done properly. Both seasons get
placed against the players who actually played alongside them.

In [2]:
from gambeta import level

per90 = seasons[seasons["minutes"] >= 900].copy()
per90["npg_p90"] = per90["npg"] / (per90["minutes"] / 90)

scored = level.zscore(per90, ["npg_p90"])

pair = scored[
    ((scored["player"].str.contains("Ronaldo", na=False)) & (scored["season"] == "0203"))
    | ((scored["player"].str.contains("Haaland", na=False)) & (scored["season"] == "2223"))
]
pair[["player", "season", "league", "npg", "minutes", "npg_p90", "npg_p90_z"]].round(2)

,player,season,league,npg,minutes,npg_p90,npg_p90_z
7590,Erling Haaland,2223,ENG-Premier League,29.0,2769.0,0.94,5.35
9310,Ronaldo,0203,ESP-La Liga,23.0,2422.0,0.85,4.96


The z-score answers the question the raw count could not. Note that it is
computed on non-penalty goals per 90 rather than total goals. Penalties measure
who gets assigned them, and per-90 removes the advantage of simply being picked
more often.

Now the second detail, and it matters more than the first.

## How much of a season's score is signal?

A player with 200 minutes and two goals has a per-90 rate that would lead the
league over a full season. He has not led the league. He has played two hours.

**Reliability** is the word for the share of a measurement that is signal rather
than noise, and it has an operational test: a reliable measurement predicts the
next one, an unreliable measurement does not. So I bucket seasons by minutes
played and ask how well each bucket predicts the same player's following season.

In [3]:
ordered = scored.sort_values(["player_id", "season"]).copy()
ordered["next_z"] = ordered.groupby("player_id")["npg_p90_z"].shift(-1)
consecutive = ordered.dropna(subset=["next_z"])

buckets = pd.cut(
    consecutive["minutes"],
    [900, 1500, 2100, 2700, 10000],
    labels=["900-1500", "1500-2100", "2100-2700", "2700+"],
)
reliability = (
    consecutive.assign(bucket=buckets)
    .groupby("bucket", observed=True)
    .apply(
        lambda g: pd.Series(
            {"seasons": len(g), "predicts next season": g["npg_p90_z"].corr(g["next_z"])}
        ),
        include_groups=False,
    )
    .round(3)
)
print(reliability)
print("\nA short season predicts the next one far worse than a long one does.")
print("That gap is noise, and it is what shrinkage exists to discount.")

           seasons  predicts next season
bucket                                  
900-1500    6681.0                 0.646
1500-2100   7855.0                 0.710
2100-2700   8435.0                 0.755
2700+       6821.0                 0.822

A short season predicts the next one far worse than a long one does.
That gap is noise, and it is what shrinkage exists to discount.


## Weighting, not shrinking

There are two ways to stop a two-hour hot streak from carrying a career, and
they are easy to confuse because they pull in the same direction.

**Shrink the value.** Pull the season's score toward the mean in proportion to
how little evidence it carries: $\tilde{z}_i = z_i \cdot m_i / (m_i + m_0)$.
The number itself gets smaller.

**Weight the value.** Leave the score alone, and give it less say when the
career is pooled: $\bar{z} = \sum_i m_i z_i / \sum_i m_i$. The number stays,
its influence shrinks.

This project does the second one. `gate.career_profile` pools every season a
player has with `np.average(..., weights=minutes)`, so a 400-minute campaign
enters the career at roughly an eighth the weight of a full one.

I want to be direct about how I know that, because for a while I did not. This
book used to teach the first method, with a prior of 900 minutes, and the
library had the function to do it. Nothing ever called it. The sensitivity
chapter is where that came out, by sweeping the prior from zero to ten thousand
and watching the ranking not move.

In [4]:
from gambeta import needs

career = pd.DataFrame(
    {
        "player_id": ["demo"] * 4,
        "player": ["A Player"] * 4,
        "league": ["ENG-Premier League"] * 4,
        "season": ["1920", "2021", "2122", "2223"],
        "minutes": [400, 900, 1800, 3200],
        **{r.key: [2.5, 2.5, 2.5, 2.5] for r in needs.OUTFIELD if r.kind == "season"},
    }
)

weights = career["minutes"] / career["minutes"].sum()
pd.DataFrame(
    {
        "minutes": career["minutes"],
        "season z": 2.5,
        "share of the career": weights.round(3),
        "contribution": (2.5 * weights).round(3),
    }
)

,minutes,season z,share of the career,contribution
0,400,2.5,0.063,0.159
1,900,2.5,0.143,0.357
2,1800,2.5,0.286,0.714
3,3200,2.5,0.508,1.270


Four identical performances. The score column never changes, which is the whole
point: the 400-minute season still says *this player performed at 2.5 sigma*,
because he did. What changes is that it supplies 6% of the career rather than
25%.

That is a weaker correction than shrinkage, and weaker in a specific way worth
naming. Under shrinkage, four seasons of 2.5 sigma at 400, 900, 1800 and 3200
minutes would pool to something below 2.5, because every input was pulled down
first. Under weighting they pool to exactly 2.5, because a weighted average of
identical numbers is that number.

So the project protects against a short season **dominating** a career. It does
not protect against a short season **being wrong**.

In [5]:
short = scored[(scored["minutes"] < 1200) & (scored["npg_p90_z"] > 2.5)]
print(f"{len(short):,} seasons under 1,200 minutes score above 2.5 sigma\n")
print(
    short.nlargest(6, "npg_p90_z")[["player", "season", "league", "minutes", "npg_p90_z"]]
    .round(2)
    .to_string(index=False)
)
print("\nEach enters its career at full strength, weighted only by its minutes.")

168 seasons under 1,200 minutes score above 2.5 sigma

           player season             league  minutes  npg_p90_z
  Viorel Moldovan   0304        FRA-Ligue 1    941.0       6.61
             Fred   0607        FRA-Ligue 1   1143.0       6.21
      Gareth Bale   2021 ENG-Premier League    920.0       6.14
  Michy Batshuayi   1415        FRA-Ligue 1    911.0       5.90
   Erling Haaland   1920     GER-Bundesliga   1063.0       5.60
Toifilou Maoulida   0910        FRA-Ligue 1   1064.0       5.55

Each enters its career at full strength, weighted only by its minutes.


Those are the seasons a shrinkage prior would have pulled hardest, and this
project leaves them alone.

There is one place where even the weighting does not reach. `consistency` is a
player's twentieth-percentile season, and a percentile does not take weights, so
a short lucky campaign counts exactly as much as a full one when the floor of a
career is worked out. That is a real gap and it is not currently closed.

## 5. Assumptions

1. **A league-season is a closed population.** Nobody outside it is relevant to
   judging a performance inside it. This is what licenses $\text{ddof}=0$, and
   it is also why I never compare a player directly against another era's raw
   numbers.
2. **The distribution within a group is roughly symmetric.** A z-score is a
   sensible summary when the mean and standard deviation describe the shape. For
   goals per 90 among regulars that is approximately true. For the full squad
   including defenders it is not, which is why the minutes floor exists.
3. **Minutes are a fair proxy for evidence.** A player who played twice as long
   is treated as having produced twice as much evidence about himself. Injuries,
   rotation and tactical benching all break that individually.
4. **The 900-minute floor does the work a prior would have done.** Seasons below
   it never enter the ranking at all, which is a blunt version of the same idea:
   discard rather than discount.

## 6. How it breaks

Two ways. One is a division that cannot be done, and one is a gap the
weighting does not reach.

**A group too small to have a spread.** The standard deviation of a handful of
players is unstable, and if every player in a group has the same value it is
zero, which means dividing by zero. `level.zscore` returns 0 rather than
infinity, which is the safe answer, but it is worth seeing that a "perfectly
average" score can actually mean "we could not tell".

In [6]:
tiny = pd.DataFrame(
    {
        "league": ["TEST"] * 3,
        "season": ["0001"] * 3,
        "value": [1.0, 1.0, 1.0],
    }
)
print("three players, identical output, zero spread:")
print(level.zscore(tiny, ["value"])[["value", "value_z"]])
print("\nz = 0 here means 'no information', not 'exactly average ability'.")

three players, identical output, zero spread:
   value  value_z
0    1.0      0.0
1    1.0      0.0
2    1.0      0.0

z = 0 here means 'no information', not 'exactly average ability'.


**A weighted average cannot rescue a percentile.** The gap named above is worth
seeing rather than asserting, because it is the one place a two-hour hot streak
still gets a full vote.

In [7]:
qualified = ranking[ranking["qualified"]].head(40)
sample = seasons[seasons["player_id"].isin(qualified["player_id"])]

thin = sample[sample["minutes"] < 1500]
print(f"among the top 40 qualifiers, {len(thin)} seasons fall under 1,500 minutes")
print("and each is one unweighted observation when `consistency` takes a percentile.\n")
print(
    thin.nsmallest(5, "minutes")[["player", "season", "minutes", "season_score"]]
    .round(2)
    .to_string(index=False)
)

among the top 40 qualifiers, 44 seasons fall under 1,500 minutes
and each is one unweighted observation when `consistency` takes a percentile.

       player season  minutes  season_score
   Phil Foden   1920    901.0          0.72
 Lionel Messi   0506    911.0          1.18
   Edin Džeko   1415    932.0          0.34
Mohamed Salah   1415    941.0          0.45
         Kaká   1213    954.0         -0.17


The honest summary: **this project discards short seasons at a hard floor and
then weights whatever survives, rather than discounting anything smoothly.**

That is a coarser instrument than empirical-Bayes shrinkage and it has one clear
advantage, which is that nothing is invented. A season either counts or it does
not, and when it counts it says what it measured. The cost is the percentile
gap above, and a 950-minute season that is trusted exactly as much per minute as
a 3,400-minute one.

Closing that properly is not a matter of picking a better prior. It is the
hierarchical model in `PENDING.md`, which would estimate how much to trust each
season from the data instead of from a constant I chose.